In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

In [2]:
interactions = pd.read_csv("./data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("./data_final_project/KuaiRec/data/small_matrix.csv")
item_features = pd.read_csv("./data_final_project/KuaiRec/data/item_daily_features.csv")

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    return df

item_features = clean_df(item_features)
item_features = item_features.drop_duplicates(subset='video_id')
train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

In [3]:
item_features['upload_dt'] = pd.to_datetime(item_features['upload_dt'])
item_features['date'] = pd.to_datetime(item_features['date'])
item_features['video_age'] = (item_features['date'] - item_features['upload_dt']).dt.days
item_features['is_short_video'] = (item_features['video_duration'].fillna(0) <= 30).astype(int)

We can drop from item_features :
- play_duration => it is the same as watch ratio

In [4]:
correlation = item_features[[
       'video_duration', 'video_width',
       'video_height', 'music_id',
       'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
       'play_duration', 'complete_play_cnt', 'complete_play_user_num',
       'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
       'long_time_play_user_num', 'short_time_play_cnt',
       'short_time_play_user_num', 'play_progress', 'comment_stay_duration',
       'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       'cancel_collect_user_num']].corr()

upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]
# These columns are dropped as we don't need them anymore 
# (merged in previous cell or just not needed for collaborative-filtering)
to_drop.extend(['date', 'upload_dt', 'video_duration', 'music_id', 'video_height', 'video_width','video_tag_name', 'play_progress'])
item_features.drop(columns=to_drop, inplace=True, errors='ignore')


In [46]:
item_features.columns

Index(['video_id', 'author_id', 'video_type', 'upload_type', 'visible_status',
       'video_tag_id', 'show_cnt', 'play_progress', 'comment_stay_duration',
       'cancel_like_cnt', 'comment_cnt', 'reply_comment_cnt',
       'comment_like_cnt', 'cancel_follow_cnt', 'share_cnt', 'report_cnt',
       'video_age', 'is_short_video'],
      dtype='object')

In [5]:
train_df = pd.merge(train_df, item_features, on='video_id', how='left')
test_df = pd.merge(test_df, item_features, on='video_id', how='left')

In [43]:
train_df.columns

Index(['user_id', 'video_id', 'play_duration', 'video_duration', 'time',
       'date', 'timestamp', 'watch_ratio', 'author_id', 'video_type',
       'upload_type', 'visible_status', 'video_width', 'video_height',
       'music_id', 'video_tag_id', 'video_tag_name', 'show_cnt',
       'play_progress', 'comment_stay_duration', 'cancel_like_cnt',
       'comment_cnt', 'reply_comment_cnt', 'comment_like_cnt',
       'cancel_follow_cnt', 'share_cnt', 'report_cnt', 'video_age',
       'is_short_video'],
      dtype='object')

In [ ]:
def build_engagement_score(df):
    df["engagement_score"] = (
        #df['is_short_video'].fillna(0) * 5 +
        df['watch_ratio'].fillna(0) #* 4 
        # + df['play_progress'].fillna(0) * 2 == watch ratio
    )
    # Video_type
    df["engagement_score"] += np.where(
        df['video_type'] == 'AD',
        -5,  # malus for ads
        5    # bonus for normal videos
    )

    # Visible_status
    df["engagement_score"] += np.where(
        df['visible_status'] == 'public',
        5,  # bonus for public videos
        -5  # malus for private and only_friends
    )

    # Upload_type
    upload_type_weights = {
        'ShortImport': 5,
        'StartCamera': 4,
        'Knowle': 3,
        'Web': 2,
        'LongImport': 1,
        'UNKNOWN': 0,
        'LongCamera': 0,
        'PictureSet': 0,
        'LongPicture': 0,
        'ACurlVideo': 0,
        'followShot': 0,
        'ShareFromOtherApp': 0,
        'SameFrame': 0,
        'PictureCopy': 0,
        'FlashPhoto': 0,
        'PhotoCopy': 0,
        'LocalCollection': 0,
        'LocalInteraction': 0
    }
    df["engagement_score"] += df['upload_type'].map(upload_type_weights).fillna(0)

    return df

test_df = build_engagement_score(test_df)
train_df = build_engagement_score(train_df)


In [7]:
# Get Unique user and videos
user_ids_train = train_df['user_id'].unique()
video_ids_train = train_df['video_id'].unique()

# Compute index for each user and videos
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids_train)}
video_to_index = {video_id: idx for idx, video_id in enumerate(video_ids_train)}

# add the index to the train and test
train_df['user_index'] = train_df['user_id'].map(user_to_index)
train_df['video_index'] = train_df['video_id'].map(video_to_index)

test_df['user_index'] = test_df['user_id'].map(user_to_index)
test_df['video_index'] = test_df['video_id'].map(video_to_index)

In [8]:
row = train_df['user_index'].values
col = train_df['video_index'].values

data = train_df['engagement_score'].values

n_users = train_df['user_index'].max() + 1
n_items = train_df['video_index'].max() + 1
    
user_item_matrix = csr_matrix((data, (row, col)), shape=(n_users, n_items))

In [9]:
R = (user_item_matrix != 0).astype(float)

def normalize_ratings(Y, R):
    Ymean = np.zeros(Y.shape[0])
    for i in range(Y.shape[0]):
        if np.sum(R[i, :]) > 0:  # Check if user has any ratings
            Ymean[i] = np.sum(Y[i, :] * R[i, :]) / np.sum(R[i, :])
    
    Ynorm = np.zeros_like(Y)
    for i in range(Y.shape[0]):
        Ynorm[i, :] = (Y[i, :] - Ymean[i]) * R[i, :]
        
    return Ynorm, Ymean

Y_dense = user_item_matrix.toarray()
R_dense = R.toarray()

Ynorm, Ymean = normalize_ratings(Y_dense, R_dense)
user_item_matrix = csr_matrix(Ynorm)

In [10]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=15,
    regularization=0.2,
    iterations=15,
    use_gpu=False,
    alpha=10
)

model.fit(user_item_matrix.T) 

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.05410599708557129 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

In [11]:
top_n=100
def get_top_n_recommendations(model, user_item_matrix, user_ids, n=10, seen=True):
    recommendations = {}
    
    for user_id in user_ids:
        # Items the user has already interacted with (adjusted for training)
        already_interacted = set(user_item_matrix[user_id].indices) if seen else set()
        
        # U * V^T
        scores = model.user_factors[user_id].dot(model.item_factors.T) + Ymean[user_id]
       
        item_scores = [(item_id, scores[item_id])
                       for item_id in range(len(scores))
                       if item_id not in already_interacted]
        # Sort and select top-N items
        item_scores.sort(key=lambda x: x[1], reverse=True)
        top_items = [item[0] for item in item_scores[:n]]
        
        recommendations[user_id] = top_items
    print(item_scores[::-1])
    return recommendations

train_users = train_df['user_index'].unique()
test_users = test_df['user_index'].unique()

# For training eval, exclude seen items to simulate a true recommendation
train_recommendations = get_top_n_recommendations(
    model, user_item_matrix, train_users, n=top_n, seen=False  # Exclude seen items in training
)

# For test eval, include only unseen items (as per real-world recommendation)
test_recommendations = get_top_n_recommendations(
    model, user_item_matrix, test_users, n=top_n, seen=True  # Exclude seen items in test (real-world)
)

[(4224, np.float64(1.0792674707700562)), (7131, np.float64(1.114904611044104)), (4100, np.float64(1.143150268726523)), (3133, np.float64(1.1564749168683839)), (1975, np.float64(1.168196051054175)), (6408, np.float64(1.2121037232209992)), (1231, np.float64(1.230320243768866)), (6747, np.float64(1.2351573395063233)), (374, np.float64(1.2378459560205293)), (1008, np.float64(1.237955241136725)), (3370, np.float64(1.243100254469092)), (4195, np.float64(1.2437989699174714)), (5095, np.float64(1.2457322167684388)), (987, np.float64(1.258371113472159)), (6535, np.float64(1.2634662079145265)), (4744, np.float64(1.2647594498922181)), (1255, np.float64(1.2648253428270173)), (6465, np.float64(1.2671987401773286)), (5821, np.float64(1.2706441568662477)), (4142, np.float64(1.2718054937650514)), (936, np.float64(1.273322968416388)), (2785, np.float64(1.2768793391515565)), (5536, np.float64(1.2785276161958528)), (1246, np.float64(1.279390423231299)), (2849, np.float64(1.2794886516858888)), (4096, np.f

In [30]:
print(test_recommendations)

{np.int64(14): [1279, 1699, 5567, 1231, 6243, 737, 2959, 3951, 3684, 6716, 3975, 4921, 1399, 1402, 1092, 516, 4096, 921, 544, 738, 1012, 5565, 5017, 4042, 3705, 2900, 1956, 6566, 3419, 2752, 5341, 1098, 863, 6106, 1210, 1108, 5137, 6852, 5215, 1227, 7175, 2535, 5922, 3074, 3537, 4014, 2855, 2940, 5389, 1188, 6353, 3698, 3923, 1078, 2558, 2308, 6743, 179, 496, 2466, 5690, 6793, 4159, 3522, 3329, 716, 5504, 6820, 6496, 4089, 1938, 6670, 4436, 77, 95, 829, 6156, 6480, 3856, 5057, 4486, 5453, 4751, 4145, 358, 1967, 789, 3197, 2989, 1548, 6446, 1189, 4252, 4895, 3172, 4400, 230, 3969, 3416, 2023], np.int64(19): [2271, 4096, 4145, 1188, 6145, 4052, 737, 5690, 6469, 1231, 921, 5922, 1697, 5389, 863, 2299, 6243, 6793, 5060, 1279, 7050, 3951, 3197, 2938, 1938, 1422, 2613, 326, 4159, 95, 6398, 4779, 7105, 3775, 5168, 2855, 1975, 621, 2752, 1326, 3329, 333, 1078, 4579, 1092, 1402, 7175, 4269, 2874, 4921, 5050, 1012, 613, 4464, 1466, 6496, 1860, 5215, 7082, 3576, 3172, 3802, 4092, 3238, 2377, 6294

In [12]:
def evaluate_recommendations(recommendations, test_df, top_n=10):
    # Map of actual items per user
    user_actual_items = test_df.groupby('user_index')['video_index'].apply(set).to_dict()
    
    precision_at_n = []
    recall_at_n = []

    for user_id, recommended_items in recommendations.items():
        if user_id in user_actual_items:
            actual_items = user_actual_items[user_id]
            recs_at_n = recommended_items[:top_n]

            num_relevant = len(set(recs_at_n) & actual_items)
            precision = num_relevant / len(recs_at_n) if recs_at_n else 0
            recall = num_relevant / len(actual_items) if actual_items else 0

            precision_at_n.append(precision)
            recall_at_n.append(recall)

    avg_precision = np.mean(precision_at_n) if precision_at_n else 0
    avg_recall = np.mean(recall_at_n) if recall_at_n else 0

    return avg_precision, avg_recall

print("Evaluating on training set...")
train_precision, train_recall = evaluate_recommendations(train_recommendations, train_df, top_n=top_n)
print(f"Training - Precision@{top_n}: {train_precision:.4f}, Recall@{top_n}: {train_recall:.4f}")

print("Evaluating on test set...")
test_precision, test_recall = evaluate_recommendations(test_recommendations, test_df, top_n=top_n)
print(f"Testing - Precision@{top_n}: {test_precision:.4f}, Recall@{top_n}: {test_recall:.4f}")

Evaluating on training set...
Training - Precision@100: 0.2035, Recall@100: 0.0140
Evaluating on test set...
Testing - Precision@100: 0.4726, Recall@100: 0.0148


In [13]:
print("\nEvaluating at different recommendation list lengths:")
for n in [5, 10, 20, 50, 100]:
    test_precision, test_recall = evaluate_recommendations(test_recommendations, test_df, top_n=n)
    print(f"Test - Precision@{n}: {test_precision:.4f}, Recall@{n}: {test_recall:.4f}")


Evaluating at different recommendation list lengths:
Test - Precision@5: 0.4700, Recall@5: 0.0007
Test - Precision@10: 0.4801, Recall@10: 0.0015
Test - Precision@20: 0.4829, Recall@20: 0.0030
Test - Precision@50: 0.4758, Recall@50: 0.0075
Test - Precision@100: 0.4726, Recall@100: 0.0148


In [57]:
def hit_rate_at_k(recommendations, ground_truth, k):
    hits = 0
    for user, recs in recommendations.items():
        gt = ground_truth.get(user, set())
        if any(item in gt for item in recs[:k]):
            hits += 1
    return hits / len(ground_truth) if ground_truth else 0

def mrr_at_k(recommendations, ground_truth, k):
    mrr = 0.0
    for user, recs in recommendations.items():
        gt = ground_truth.get(user, set())
        for rank, item in enumerate(recs[:k], start=1):
            if item in gt:
                mrr += 1.0 / rank
                break
    return mrr / len(ground_truth) if ground_truth else 0

def ndcg_at_k(recommendations, ground_truth, k):
    def dcg(recs, gt, k):
        return sum((1 / np.log2(i + 2)) if rec in gt else 0 for i, rec in enumerate(recs[:k]))

    def idcg(gt, k):
        n_relevant = min(len(gt), k)
        return sum(1 / np.log2(i + 2) for i in range(n_relevant))

    total_ndcg = 0.0
    count = 0
    for user, recs in recommendations.items():
        gt = ground_truth.get(user, set())
        idcg_val = idcg(gt, k)
        if idcg_val == 0:
            continue
        total_ndcg += dcg(recs, gt, k) / idcg_val
        count += 1
    return total_ndcg / count if count > 0 else 0

# Create test user-item matrix for ground truth
test_row = test_df['user_index'].values
test_col = test_df['video_index'].values
test_data = np.ones(len(test_df))  # binary implicit feedback

test_user_item_matrix = csr_matrix((test_data, (test_row, test_col)), shape=(n_users, n_items))

def sparse_matrix_to_dict(matrix):
    user_item_dict = {}
    for user_id in range(matrix.shape[0]):
        items = matrix[user_id].indices
        if len(items) > 0:
            user_item_dict[user_id] = set(items)
    return user_item_dict

# Get ground truth from test matrix
ground_truth = sparse_matrix_to_dict(test_user_item_matrix)

# Evaluate with additional metrics
k = 10
print("Hit Rate@10:", hit_rate_at_k(test_recommendations, ground_truth, k))
print("MRR@10:", mrr_at_k(test_recommendations, ground_truth, k))
print("nDCG@10:", ndcg_at_k(test_recommendations, ground_truth, k))

Hit Rate@10: 0.9978738483345145
MRR@10: 0.6552203210601526
nDCG@10: 0.47710152271176964
